Ce notebook vous guide à travers les bases de l'utilisation de l'API Gemini avec le SDK Python, en couvrant des sujets tels que l'installation, l'authentification, les configurations de génération de contenu (température, top-P, longueur de sortie) et différentes techniques de "prompting" (zero-shot, few-shot, chaîne de pensée, ReAct, mode réflexion, génération et explication de code).

## Démarrer avec Google Colab

Ce notebook a été conçu à l'origine pour Kaggle ; cette version est adaptée pour **Google Colab**.

1. Ouvrez le fichier dans Colab : **Fichier > Importer le notebook** (ou déposez le fichier `.ipynb` dans Google Drive puis ouvrez-le avec Colab).
2. Enregistrez une copie dans votre Drive : **Fichier > Enregistrer une copie dans Drive**.

Cette section initiale explique comment configurer et utiliser le notebook dans Google Colab, en soulignant l'importance d'enregistrer une copie et de suivre l'ordre d'exécution des cellules.

Chaque cellule de code dispose d'un bouton ▶️ **Exécuter** à sa gauche (ou utilisez `Maj + Entrée`). Ces notebooks sont prévus pour être exécutés dans l'ordre, de haut en bas, mais n'hésitez pas à ajouter de nouvelles cellules, à exécuter votre propre code et à explorer. Si vous êtes bloqué, utilisez **Exécution > Redémarrer la session** (ou **Déconnecter et supprimer l'environnement d'exécution**), puis relancez les cellules depuis le début.



## Démarrer avec l'API Gemini

Tous les exercices de ce notebook utilisent l'[API Gemini](https://ai.google.dev/gemini-api/) via le [SDK Python](https://pypi.org/project/google-genai/). Chacun de ces prompts est également accessible directement dans [Google AI Studio](https://aistudio.google.com/) : si vous préférez utiliser une interface web et vous passer du code pour cette activité, cherchez le lien <img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> AI Studio sous chaque prompt.

Cette partie présente l'objectif principal du notebook : explorer l'API Gemini via son SDK Python. Elle mentionne également Google AI Studio comme une alternative pour les utilisateurs préférant une interface web.

Ensuite, vous devez ajouter votre clé API dans Colab sous forme de **secret Colab** :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**.
3. Dans **Nom**, saisissez exactement `GOOGLE_API_KEY` ; dans **Valeur**, collez votre clé API.
4. Activez l'interrupteur **Accès au notebook** pour ce secret.

Les secrets Colab sont liés à votre compte Google : ils ne sont pas enregistrés dans le notebook et ne sont pas partagés si vous partagez le fichier.

Avant de pouvoir interagir avec l'API Gemini, il est nécessaire d'installer le SDK Python correspondant. La cellule suivante utilise `pip` pour installer ou mettre à jour le package `google-genai`.

### Installer le SDK

In [4]:
# Installation ou mise à jour du SDK officiel Google GenAI pour interagir avec Gemini
!pip install -U -q "google-genai"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


Cette cellule importe les bibliothèques nécessaires : `genai` pour l'accès à l'API Gemini et `IPython.display` pour améliorer l'affichage des résultats dans le notebook, notamment pour le Markdown.

Importez le SDK ainsi que quelques utilitaires pour afficher les résultats.

In [5]:
# Importation de la bibliothèque principale pour interagir avec l'API Gemini
from google import genai
# Importation des types de configuration nécessaires (ex: GenerateContentConfig)
from google.genai import types

# Importation d'utilitaires pour un affichage propre des formats HTML et Markdown dans Colab
from IPython.display import HTML, Markdown, display

Pour gérer les limites de quota de l'API (qui peuvent survenir lors d'appels fréquents), un mécanisme de nouvelle tentative est mis en place. Cela permet au notebook de retenter automatiquement les requêtes en cas d'erreur de quota (429) ou de serveur (503), améliorant ainsi la robustesse de l'exécution.

Mettez en place un mécanisme de nouvelle tentative (retry). Il vous permet de lancer « Tout exécuter » sans vous soucier du quota par minute.

In [6]:
# Importation du module permettant de gérer automatiquement les limites de quota (retries)
from google.api_core import retry

# Définition de la règle : on réessaye en cas d'erreurs 429 (Trop de requêtes) ou 503 (Service indisponible)
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

# Application automatique de la politique de retry sur la fonction generate_content
genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

Cette section est cruciale pour l'authentification. Elle explique comment obtenir et configurer votre clé API Google, en priorisant les secrets Colab, puis les secrets Kaggle, et enfin les variables d'environnement. La clé API est indispensable pour toutes les interactions avec l'API Gemini.

### Configurer votre clé API

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` (voir les étapes ci-dessus).

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

La cellule ci-dessous lit d'abord le secret Colab. Par commodité, elle essaie aussi les secrets Kaggle puis la variable d'environnement `GOOGLE_API_KEY`, ce qui permet d'utiliser le même notebook ailleurs.

In [7]:
import os

# Initialisation de la clé d'API
GOOGLE_API_KEY = "GOOGLE_API_KEY"

# 1) Tentative de récupération depuis les secrets de Google Colab (icône clé dans le menu gauche)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass

# 2) Alternative : Récupération depuis les secrets si exécuté sur Kaggle
if not GOOGLE_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass

# 3) Alternative : Récupération via les variables d'environnement (ex: exécution locale)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

# Si aucune clé n'est trouvée, une erreur explicite est levée
if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

print("Clé API chargée de manière sécurisée ✅")

Clé API chargée de manière sécurisée ✅


Après avoir configuré la clé API, cette partie montre comment effectuer votre première requête à l'API Gemini. Elle initialise un client avec la clé API et utilise le modèle spécifié pour générer une explication simple de l'IA. La réponse est ensuite affichée dans la console.

Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

### Lancer votre premier prompt

1.   List item
2.   List item



Dans cette étape, vous allez vérifier que votre clé API est bien configurée en envoyant une requête.

Le SDK Python utilise un [objet `Client`](https://googleapis.github.io/python-genai/genai.html#genai.client.Client) pour envoyer les requêtes à l'API. Le client permet de choisir le back-end utilisé (API Gemini ou Vertex AI) et gère l'authentification (la clé API).

**Remarque** : si une `TransportError` apparaît à cette étape, redémarrez la session une fois (**Exécution > Redémarrer la session**).

In [12]:
# Définition du modèle recommandé mis à jour pour votre compte
MODEL = "gemini-3.5-flash"

# Initialisation du client de l'API Gemini avec notre clé d'authentification
client = genai.Client(api_key=GOOGLE_API_KEY)

# Envoi d'un premier prompt simple (One-shot) pour expliquer l'IA à un enfant
response = client.models.generate_content(
    model=MODEL,
    contents="Explain AI to me like I'm a kid."
)

# Affichage de la réponse en texte brut
print(response.text)

Imagine you have a puppy. When you first get him, he doesn't know how to "sit" or "fetch." You have to teach him by showing him what to do and giving him treats when he gets it right. 

**AI (which stands for Artificial Intelligence) is like a puppy made of computer code.** 

It’s not a physical robot (though it can live inside one). It’s a computer program that can learn to do things on its own, without a human having to program every single step.

Here is how it works:

### 1. How does AI learn? (The "Dog" Game)
If you want to teach an AI what a dog looks like, you can't just say, *"It has four legs and fur,"* because a cat has those too! 

Instead, you show the computer **millions of pictures** of dogs. 
* "This is a dog." 
* "This is a cat (not a dog)." 
* "This is a loaf of bread (definitely not a dog)."

At first, the AI makes silly mistakes. But after looking at millions of pictures, it starts to notice patterns (like the shape of a dog's nose or ears). Soon, it becomes a master

Le modèle Gemini renvoie souvent son contenu au format Markdown. Cette cellule utilise la fonction `Markdown` d'IPython.display pour rendre la réponse de manière plus lisible et formatée directement dans le notebook.

La réponse est souvent renvoyée au format Markdown, que vous pouvez afficher directement dans ce notebook.

In [13]:
# Conversion et affichage de la réponse brute au format Markdown visuel riche
Markdown(response.text)

Imagine you have a puppy. When you first get him, he doesn't know how to "sit" or "fetch." You have to teach him by showing him what to do and giving him treats when he gets it right. 

**AI (which stands for Artificial Intelligence) is like a puppy made of computer code.** 

It’s not a physical robot (though it can live inside one). It’s a computer program that can learn to do things on its own, without a human having to program every single step.

Here is how it works:

### 1. How does AI learn? (The "Dog" Game)
If you want to teach an AI what a dog looks like, you can't just say, *"It has four legs and fur,"* because a cat has those too! 

Instead, you show the computer **millions of pictures** of dogs. 
* "This is a dog." 
* "This is a cat (not a dog)." 
* "This is a loaf of bread (definitely not a dog)."

At first, the AI makes silly mistakes. But after looking at millions of pictures, it starts to notice patterns (like the shape of a dog's nose or ears). Soon, it becomes a master at spotting dogs!

### 2. Where do you see AI every day?
You probably use AI every day without even knowing it!
* **YouTube or Netflix:** When it suggests a video you might like, that’s an AI guessing your favorite things.
* **Video Games:** When you play a game and the bad guys chase you or dodge your attacks, that’s AI making them "smart."
* **Siri or Alexa:** When you talk to a smart speaker, an AI listens to your voice, figures out what your words mean, and talks back to you.
* **Funny Filters:** When you use a Snapchat or TikTok filter that puts bunny ears on your head, an AI is finding where your face is.

### 3. The Big Secret: Is AI alive?
**Nope!** AI doesn't have feelings, it can’t think for itself, and it doesn't eat pizza. 

It is just a super-fast calculator. It looks at a lot of information, finds patterns, and makes really good guesses. 

So, AI is like a very helpful, super-smart computer sidekick!

Au-delà des requêtes simples à un seul tour, l'API Gemini supporte des conversations à plusieurs tours. Cette section initialise une session de chat et démontre comment envoyer des messages successifs tout en conservant le contexte de la conversation.

### Démarrer une conversation

L'exemple précédent utilise une structure à un seul tour (texte en entrée, texte en sortie), mais vous pouvez aussi mettre en place une conversation à plusieurs tours.

In [14]:
# Création d'une session de chat interactive qui mémorise automatiquement l'historique
chat = client.chats.create(model=MODEL, history=[])

# Premier message : introduction de l'utilisateur
response = chat.send_message('Hello! My name is Zlork.')
print(response.text)

Hello, Zlork! It's great to meet you. How can I help you today?


Cette cellule poursuit la conversation en envoyant un nouveau message. Le modèle utilise l'historique de la conversation pour répondre de manière contextuellement pertinente.

In [ ]:
# Envoi d'un deuxième message dans la même session de chat (le modèle conserve le contexte de Zlork)
response = chat.send_message('Can you tell me something interesting about dinosaurs?')
print(response.text)

Here is one of my favorite mind-bending facts about them:

**The *Tyrannosaurus rex* lived closer in time to modern humans than it did to the *Stegosaurus*.**

We often picture all dinosaurs sharing the Earth at the same time, but the "Age of Dinosaurs" lasted for an unfathomably long time—about 180 million years! 

* The **Stegosaurus** walked the Earth during the late Jurassic period, around **150 million years ago**.
* The **T. rex** didn't evolve until the late Cretaceous period, around **66 million years ago**.

That means there is an **84-million-year gap** between *Stegosaurus* and *T. rex*, but only a **66-million-year gap** between *T. rex* and you! To a *T. rex*, a *Stegosaurus* was an ancient, long-fossilized relic of the past.

***

**Bonus Fact:** We actually know the true color of a few dinosaurs! 
In recent years, paleontologists discovered microscopic structures called **melanosomes** (which hold pigment) preserved in fossilized feathers. Because of this, we know that a

Pour vérifier que le contexte de la conversation est bien maintenu, cette cellule demande au modèle de rappeler le nom mentionné précédemment. Cela illustre la capacité du modèle à conserver une mémoire à court terme dans le cadre d'une session de chat.

L'API Gemini propose différents modèles. Cette section montre comment lister tous les modèles disponibles via l'API, ce qui est utile pour comprendre quelles options sont à votre disposition et leurs noms exacts.

Tant que l'objet `chat` existe, l'état de la conversation est conservé. Vérifiez-le en demandant au modèle s'il connaît le nom de l'utilisateur.

In [ ]:
# Question test pour vérifier que la mémoire de session fonctionne toujours
response = chat.send_message('Do you remember what my name is?')
print(response.text)

Yes, of course! Your name is **Zlork**.


### Choisir un modèle

L'API Gemini donne accès à plusieurs modèles de la famille Gemini. Découvrez les modèles disponibles et leurs capacités sur la [page de présentation des modèles](https://ai.google.dev/gemini-api/docs/models/gemini).

Dans cette étape, vous allez utiliser l'API pour lister tous les modèles disponibles.

In [ ]:
# Boucle sur tous les modèles actuellement disponibles via votre clé API
for model in client.models.list():
  print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini

En plus de lister les noms, il est possible d'obtenir des informations détaillées sur chaque modèle, telles que les limites de jetons, la température maximale, et d'autres capacités. Cette cellule filtre les informations pour le modèle utilisé (`MODEL`) et les affiche de manière structurée.

La réponse de [`models.list`](https://ai.google.dev/api/models#method:-models.list) renvoie aussi des informations supplémentaires sur les capacités de chaque modèle, comme les limites de jetons (tokens) et les paramètres pris en charge.

In [ ]:
from pprint import pprint

# Recherche et affichage complet des détails techniques spécifiques à notre modèle (MODEL)
for model in client.models.list():
  if model.name == f'models/{MODEL}':
    # to_json_dict() permet de convertir les métadonnées du modèle en dictionnaire lisible
    pprint(model.to_json_dict())
    break

{'description': 'Gemini 3.8 Flash',
 'display_name': 'Gemini 3.8 Flash',
 'input_token_limit': 1048576,
 'max_temperature': 2.0,
 'name': 'models/gemini-3.8-flash',
 'output_token_limit': 65536,
 'supported_actions': ['generateContent',
                       'countTokens',
                       'createCachedContent',
                       'batchGenerateContent'],
 'temperature': 1.0,
 'thinking': True,
 'top_k': 64,
 'top_p': 0.95,
 'tuned_model_info': {},
 'version': '3.0'}


Cette section explore les paramètres de génération de contenu, qui permettent d'affiner le comportement du modèle lors de la création de texte.

## Explorer les paramètres de génération



Le paramètre `max_output_tokens` contrôle la longueur maximale de la sortie générée par le modèle. Une valeur faible peut tronquer la réponse. Cet exemple démontre l'impact de ce paramètre en demandant un essai de 1000 mots avec une limite de 200 jetons.

### Longueur de la sortie

Lorsque vous générez du texte avec un LLM, la longueur de la sortie influe sur le coût et les performances. Générer plus de jetons demande plus de calcul, ce qui augmente la consommation d'énergie, la latence et le coût.

Pour empêcher le modèle de générer des jetons au-delà d'une limite, vous pouvez indiquer le paramètre `max_output_tokens` lors de l'appel à l'API Gemini. Ce paramètre n'influence pas la manière dont les jetons sont générés : la sortie ne devient pas plus concise sur le plan du style ou du contenu, la génération s'arrête simplement une fois la longueur atteinte. Un travail sur le prompt peut être nécessaire pour obtenir une réponse complète dans la limite choisie.

> **Remarque (modèles Gemini 3)** : ces modèles « réfléchissent » avant de répondre, et les jetons de réflexion sont comptés dans `max_output_tokens`. Avec une limite très basse, la réponse peut donc être tronquée, voire vide (`None`). Dans ce cas, augmentez la limite.

In [15]:
from google.genai import types

# Configuration d'une limite stricte à 200 jetons (mots / parties de mots) en sortie
short_config = types.GenerateContentConfig(max_output_tokens=200)

# Demande d'un long essai pour forcer la troncature à cause de la limite configurée
response = client.models.generate_content(
    model=MODEL,
    config=short_config,
    contents='Write a 1000 word essay on the importance of olives in modern society.'
)

print(response.text)

### The Golden Nectar of Antiquity


Encore une démonstration de `max_output_tokens` avec un prompt différent pour montrer que le modèle s'arrête simplement une fois la limite atteinte, même si la réponse n'est pas complète ou cohérente.

In [ ]:
# Test similaire avec un poème pour voir si la coupure intervient en plein milieu du texte
response = client.models.generate_content(
    model=MODEL,
    config=short_config,
    contents='Write a short poem on the importance of olives in modern society.'
)

print(response.text)

From boardroom glass to rustic plate


La température est un paramètre qui contrôle le caractère aléatoire des sorties du modèle. Une température plus élevée (comme 2.0 ici) augmente la diversité et la créativité des réponses. Cet exemple génère plusieurs couleurs aléatoires pour illustrer la variation.

Testez avec vos propres prompts. Essayez un prompt avec une limite de sortie restrictive, puis adaptez le prompt pour qu'il fonctionne dans cette limite.

### Température

La température contrôle le degré d'aléatoire dans la sélection des jetons. Une température élevée augmente le nombre de jetons candidats parmi lesquels le prochain jeton est choisi, et peut produire des résultats plus variés. Une température basse a l'effet inverse : une température de 0 correspond à un décodage glouton (greedy decoding), qui choisit le jeton le plus probable à chaque étape.

La température ne garantit pas le caractère aléatoire, mais elle permet d'« orienter » un peu la sortie.

In [ ]:
from google.genai import types

# Configuration de la température à 2.0 (valeur maximale pour une créativité et variété maximale)
high_temp_config = types.GenerateContentConfig(temperature=2.0)

# Exécution de 5 requêtes identiques pour analyser la diversité des réponses générées
for _ in range(5):
  response = client.models.generate_content(
      model=MODEL,
      config=high_temp_config,
      contents='Pick a random colour... (respond in a single word)'
  )

  if response.text:
    print(response.text.strip(), '-' * 25)

Indigo -------------------------
Crimson -------------------------
Cerulean -------------------------
Crimson -------------------------
Crimson -------------------------


À l'inverse, une température basse (comme 0.0 ici) rend les sorties plus déterministes et prévisibles, le modèle ayant tendance à choisir les jetons les plus probables. Cet exemple devrait montrer moins de variabilité dans les couleurs choisies par le modèle.

Essayez maintenant le même prompt avec une température de zéro. La sortie n'est pas totalement déterministe, car d'autres paramètres influencent la sélection des jetons, mais les résultats auront tendance à être plus stables.

In [16]:
from google.genai import types

# Configuration de la température à 0.0 (rend la réponse hautement prévisible et cohérente)
low_temp_config = types.GenerateContentConfig(temperature=0.0)

# Envoi de 5 requêtes : avec une température de 0, le modèle devrait retourner presque toujours la même couleur
for _ in range(5):
  response = client.models.generate_content(
      model=MODEL,
      config=low_temp_config,
      contents='Pick a random colour... (respond in a single word)'
  )

  if response.text:
    print(response.text.strip(), '-' * 25)

Indigo -------------------------
Indigo -------------------------
Indigo -------------------------
Indigo -------------------------
Indigo -------------------------


Le paramètre `top_p` est un autre moyen de contrôler la diversité. Il définit un seuil de probabilité pour la sélection des jetons. Cet exemple illustre son utilisation en demandant une courte histoire, permettant au modèle une certaine créativité avec les valeurs par défaut de `temperature` et `top_p`.

### Top-P

Comme la température, le paramètre top-P sert à contrôler la diversité de la sortie du modèle.

Top-P définit un seuil de probabilité : les jetons sont ajoutés aux candidats, du plus probable au moins probable, jusqu'à ce que leur probabilité cumulée dépasse ce seuil. Un top-P de 0 équivaut généralement à un décodage glouton, et un top-P de 1 retient en général tous les jetons du vocabulaire du modèle.

Vous rencontrerez peut-être aussi le top-K dans la littérature sur les LLM. Le top-K n'était pas configurable dans la série Gemini 2.0, mais pouvait être modifié dans des modèles plus anciens. Le top-K est un entier positif qui définit le nombre de jetons les plus probables parmi lesquels choisir le jeton de sortie. Un top-K de 1 sélectionne un seul jeton, ce qui revient à un décodage glouton.


Exécutez cet exemple plusieurs fois, modifiez les réglages et observez l'effet sur la sortie.

In [17]:
from google.genai import types

# Configuration équilibrée : température classique (1.0) et top_p limitant les choix exotiques
model_config = types.GenerateContentConfig(
    temperature=1.0,
    top_p=0.95,
)

story_prompt = "You are a creative writer. Write a short story about a cat who goes on an adventure."
response = client.models.generate_content(
    model=MODEL,
    config=model_config,
    contents=story_prompt
)

print(response.text)

Barnaby was a cat of refined tastes and rigid schedules. 

His mornings at Number 4 Willow Lane were spent in a patch of sunlight on the Persian rug. His afternoons were reserved for supervising the Tall One (his human, Eleanor) from the safety of the bookshelf, and his evenings were dedicated to the consumption of salmon-flavored paté. Barnaby was a tuxedo cat, sporting a sleek black coat, a pristine white bib, and four white paws that made him look as though he were perpetually dressed for an opera.

But on a Tuesday in late October, the universe disrupted Barnaby’s schedule. 

Eleanor left the back door slightly ajar while carrying in groceries. Through the crack, a draft of wind slipped into the kitchen. It didn’t smell of floor wax or lavender soap. It smelled of damp earth, decaying leaves, pine sap, and *possibility*. 

Barnaby’s ears twitched. His pupils dilated until his eyes were two pools of midnight green. 

Then, he saw it. 

Hovering just beyond the threshold was a creatu

Cette section aborde différentes techniques de "prompting" pour guider le modèle et obtenir les réponses souhaitées.

## Prompting

Cette section reprend plusieurs prompts du chapitre pour que vous puissiez les tester directement avec l'API. Modifiez le texte pour voir comment chaque prompt se comporte avec des instructions différentes, davantage d'exemples ou tout autre changement qui vous vient à l'esprit.

Le prompting "zero-shot" consiste à donner une instruction au modèle sans aucun exemple. Ici, le modèle est invité à classer des critiques de films comme POSITIVE, NEUTRAL ou NEGATIVE, avec une `max_output_tokens` très basse pour contraindre la réponse.

### Zero-shot

Les prompts zero-shot décrivent directement la demande au modèle, sans exemple.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1gzKKgDHwkAvexG5Up0LMtl1-6jKMKe4g"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [18]:
from google.genai import types

# Configuration optimisée pour des réponses directes courtes (ex: classification)
model_config = types.GenerateContentConfig(
    temperature=0.1,
    top_p=1,
    max_output_tokens=15, # Ajusté légèrement pour s'assurer que le mot clé de classification ne soit pas coupé
)

# Prompt sans exemple (Zero-shot) demandant une évaluation sentimentale stricte
zero_shot_prompt = """Classify movie reviews as POSITIVE, NEUTRAL or NEGATIVE.
Review: "Her" is a disturbing study revealing the direction
humanity is headed if AI is allowed to keep evolving,
unchecked. I wish there were more movies like this masterpiece.
Sentiment: """

response = client.models.generate_content(
    model=MODEL,
    config=model_config,
    contents=zero_shot_prompt
)

print(response.text)

None


Le mode enum permet de contraindre la sortie du modèle à un ensemble prédéfini de valeurs, ce qui est utile pour des tâches de classification. Cette cellule définit un enum `Sentiment` et l'utilise pour s'assurer que la réponse du modèle correspond à l'une de ces catégories.

#### Mode enum

Les modèles sont entraînés à générer du texte, et même si les modèles Gemini suivent très bien les instructions, d'autres modèles produisent parfois plus de texte que souhaité. Dans l'exemple précédent, le modèle renvoie le label, mais il peut parfois le faire précéder de « Sentiment », et sans limite de jetons, il peut aussi ajouter une explication. Voir [ce prompt dans AI Studio](https://aistudio.google.com/prompts/1gzKKgDHwkAvexG5Up0LMtl1-6jKMKe4g) pour un exemple.

L'API Gemini propose un [mode enum](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Enum.ipynb) qui permet de contraindre la sortie à un ensemble fixe de valeurs.

In [19]:
import enum
from google.genai import types

# Définition d'un schéma d'énumération Python strict pour contraindre la classification
class Sentiment(enum.Enum):
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"

# Utilisation de response_mime_type et response_schema pour forcer l'API à répondre uniquement par une valeur de l'enum
response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        response_mime_type="text/x.enum",
        response_schema=Sentiment
    ),
    contents=zero_shot_prompt
)

print(response.text)

positive


Lorsque le mode enum est activé, le SDK Python tente de convertir automatiquement la sortie texte du modèle en un objet Python correspondant à l'enum. Cette cellule affiche le type et la valeur de la réponse `parsed` pour le confirmer.

Lorsque vous utilisez une sortie contrainte comme un enum, le SDK Python tente de convertir automatiquement la réponse texte du modèle en objet Python. Le résultat est stocké dans le champ `response.parsed`.

In [20]:
# Récupération directe de la valeur convertie sous forme d'objet Enum natif Python grâce au SDK
enum_response = response.parsed
print("Objet récupéré :", enum_response)
print("Type Python :", type(enum_response))

Objet récupéré : Sentiment.POSITIVE
Type Python : <enum 'Sentiment'>


Le prompting "one-shot" (un exemple) ou "few-shot" (plusieurs exemples) fournit au modèle des exemples de paires entrée-sortie. Cela l'aide à mieux comprendre le format et le type de réponse attendus. Cet exemple montre comment parser une commande de pizza en JSON avec des exemples.

### One-shot et few-shot

Fournir un exemple de la réponse attendue s'appelle un prompt « one-shot ». Lorsque vous fournissez plusieurs exemples, on parle de prompt « few-shot ».

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1jjWkjUSoMXmLvMJ7IzADr_GxHPJVV2bg"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>


In [21]:
from google.genai import types

# Prompt Few-shot fournissant deux exemples structurés de conversion d'une commande en format JSON
few_shot_prompt = """Parse a customer's pizza order into valid JSON:

EXAMPLE:
I want a small pizza with cheese, tomato sauce, and pepperoni.
JSON Response:
```
{
"size": "small",
"type": "normal",
"ingredients": ["cheese", "tomato sauce", "pepperoni"]
}
```

EXAMPLE:
Can I get a large pizza with tomato sauce, basil and mozzarella
JSON Response:
```
{
"size": "large",
"type": "normal",
"ingredients": ["tomato sauce", "basil", "mozzarella"]
}
```

ORDER:
"""

customer_order = "Give me a large with cheese & pineapple"

# Envoi combiné du prompt de structure d'apprentissage (few_shot_prompt) et de l'entrée utilisateur réelle (customer_order)
response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=1,
        max_output_tokens=250,
    ),
    contents=[few_shot_prompt, customer_order]
)

print(response.text)

```json
{
"size": "large",
"type": "normal",
"ingredients": ["cheese",


Le mode JSON est une extension du prompting qui permet de contraindre la sortie du modèle à un schéma JSON spécifique, garantissant une structure de données valide. Cette cellule définit un `TypedDict` pour une commande de pizza et l'utilise pour obtenir une sortie JSON structurée.

#### Mode JSON

Pour contrôler le schéma et vous assurer de ne recevoir que du JSON (sans autre texte ni Markdown), vous pouvez utiliser le [mode JSON](https://github.com/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb) de l'API Gemini. Il contraint le décodage du modèle : la sélection des jetons est guidée par le schéma fourni.

In [22]:
import typing_extensions as typing
from google.genai import types

# Définition de la structure attendue en sortie sous forme de dictionnaire typé
class PizzaOrder(typing.TypedDict):
    size: str
    ingredients: list[str]
    type: str

# Utilisation du mode JSON natif de l'API Gemini : garantit l'obtention d'un JSON valide respectant le schéma PizzaOrder
response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=PizzaOrder,
    ),
    contents="Can I have a large dessert pizza with apple and chocolate"
)

print(response.text)

{"size": "large", "ingredients": ["apple", "chocolate"], "type": "dessert"}


La "Chaîne de pensée" (Chain of Thought, CoT) invite le modèle à décomposer son raisonnement en étapes intermédiaires avant de donner la réponse finale. Cela améliore souvent la qualité et la fiabilité des réponses, en particulier pour des problèmes complexes. Cette cellule pose une énigme simple et demande une réponse directe.

### Chaîne de pensée (Chain of Thought, CoT)

Interroger directement un LLM peut donner des réponses rapides et économes (en jetons de sortie), mais sujettes aux hallucinations. La réponse peut « sembler » correcte (sur le plan de la langue et de la syntaxe) tout en étant fausse sur le fond et le raisonnement.

Le prompting par chaîne de pensée consiste à demander au modèle de produire les étapes intermédiaires de son raisonnement, ce qui donne généralement de meilleurs résultats, surtout combiné à des exemples few-shot. Notez que cette technique n'élimine pas complètement les hallucinations et qu'elle coûte en général plus cher, du fait du nombre accru de jetons.

Les modèles comme ceux de la famille Gemini sont entraînés à être « bavards » ou « réfléchis » et fournissent des étapes de raisonnement sans qu'on le leur demande. Pour cet exemple simple, vous pouvez donc demander au modèle d'être plus direct dans le prompt afin de forcer une réponse sans raisonnement. Relancez cette étape si le modèle a de la chance et trouve la bonne réponse du premier coup.

> **Remarque** : les modèles Gemini 3 raisonnent en interne avant de répondre ; ils trouvent donc souvent la bonne réponse même sans « étape par étape ».

In [23]:
# Enigme mathématique envoyée directement au modèle sans instructions d'étapes de calcul
prompt = """When I was 4 years old, my partner was 3 times my age. Now, I
am 20 years old. How old is my partner? Return the answer directly."""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt
)

print(response.text)

28


Cette cellule réutilise la même énigme, mais avec l'instruction `Let's think step by step`. On s'attend à ce que le modèle fournisse les étapes de calcul, ce qui réduit le risque d'hallucinations et augmente la transparence du raisonnement.

Essayez maintenant la même approche, mais en indiquant au modèle qu'il doit « réfléchir étape par étape » (« think step by step »).

In [24]:
# Enigme mathématique avec l'instruction CoT (Chain of Thought) "Let's think step by step" pour forcer la décomposition du raisonnement
prompt = """When I was 4 years old, my partner was 3 times my age. Now,
I am 20 years old. How old is my partner? Let's think step by step."""

response = client.models.generate_content(
    model=MODEL,
    contents=prompt
)

Markdown(response.text)

Let's think step by step:

1. **Find your partner's age when you were 4 years old:**
   * You were 4.
   * Your partner was 3 times your age: $4 \times 3 = 12$ years old.

2. **Calculate the age difference:**
   * Your partner is $12 - 4 = 8$ years older than you. This age difference will always remain the same.

3. **Calculate your partner's current age:**
   * Now, you are 20 years old.
   * Since your partner is 8 years older than you: $20 + 8 = 28$ years old.

*(Alternatively: $20 - 4 = 16$ years have passed. Adding 16 years to your partner's starting age of 12 also gives $12 + 16 = 28$.)*

Your partner is **28** years old.

ReAct (Reasoning and Acting) est une technique de prompting avancée qui combine le raisonnement (`Thought`) et l'action (`Action`) du modèle. Le modèle est invité à générer une pensée, puis une action (comme une recherche ou une consultation), dont le résultat est une observation, qui alimente la pensée suivante. Cette cellule définit les instructions et des exemples pour un système ReAct.

### ReAct : raisonner et agir

Dans cet exemple, vous allez exécuter un prompt ReAct directement avec l'API Gemini et effectuer vous-même les étapes de recherche. Comme ce prompt suit une structure bien définie, il existe des frameworks qui l'encapsulent dans des API plus simples à utiliser et qui effectuent automatiquement les appels d'outils, comme l'exemple LangChain du livre blanc « Prompting ».

Pour essayer avec le moteur de recherche de Wikipédia, consultez l'exemple du cookbook [Searching Wikipedia with ReAct](https://github.com/google-gemini/cookbook/blob/main/examples/Search_Wikipedia_using_ReAct.ipynb).


> Remarque : le prompt et les exemples en contexte utilisés ici proviennent de [https://github.com/ysymyth/ReAct](https://github.com/ysymyth/ReAct), publié sous [licence MIT](https://opensource.org/licenses/MIT), Copyright (c) 2023 Shunyu Yao.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/18oo63Lwosd-bQ6Ay51uGogB3Wk3H8XMO"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>


In [25]:
model_instructions = """
Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation,
Observation is understanding relevant information from an Action's output and Action can be one of three types:
 (1) <search>entity</search>, which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it
     will return some similar entities to search and you can try to search the information from those topics.
 (2) <lookup>keyword</lookup>, which returns the next sentence containing keyword in the current context. This only does exact matches,
     so keep your searches short.
 (3) <finish>answer</finish>, which returns the answer and finishes the task.
"""

example1 = """Question
Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?

Thought 1
The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.

Action 1
<search>Milhouse</search>

Observation 1
Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.

Thought 2
The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".

Action 2
<lookup>named after</lookup>

Observation 2
Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.

Thought 3
Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.

Action 3
<finish>Richard Nixon</finish>
"""

example2 = """Question
What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?

Thought 1
I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.

Action 1
<search>Colorado orogeny</search>

Observation 1
The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.

Thought 2
It does not mention the eastern sector. So I need to look up eastern sector.

Action 2
<lookup>eastern sector</lookup>

Observation 2
The eastern sector extends into the High Plains and is called the Central Plains orogeny.

Thought 3
The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.

Action 3
<search>High Plains</search>

Observation 3
High Plains refers to one of two distinct land regions

Thought 4
I need to instead search High Plains (United States).

Action 4
<search>High Plains (United States)</search>

Observation 4
The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130m).

Thought 5
High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.

Action 5
<finish>1,800 to 7,000 ft</finish>
"""

# Inventez d'autres exemples vous-même, ou parcourez https://github.com/ysymyth/ReAct/

Cet exemple initialise un chat ReAct. Il utilise `stop_sequences` pour faire en sorte que le modèle génère une seule étape de pensée/action à la fois, vous permettant d'intervenir et de simuler l'action (par exemple, une recherche sur Wikipédia) manuellement.

Pour obtenir une seule étape à la fois, en ignorant les éventuelles étapes « Observation » hallucinées, vous allez utiliser `stop_sequences` pour interrompre la génération. Les étapes sont, dans l'ordre : `Thought` (pensée), `Action` et `Observation`.

In [26]:
from google.genai import types

question = """Question
Who was the youngest author listed on the transformers NLP paper?
"""

# On configure 'stop_sequences' pour forcer le modèle à s'arrêter juste avant d'écrire l'Observation.
# C'est nous qui injecterons la vraie observation de recherche.
react_config = types.GenerateContentConfig(
    stop_sequences=["\nObservation"],
    system_instruction=model_instructions + example1 + example2,
)

# Création de la session de chat ReAct interactive
react_chat = client.chats.create(
    model=MODEL,
    config=react_config,
)

resp = react_chat.send_message(question)
print(resp.text)

Thought 1
I need to find the paper that introduced the Transformer architecture in NLP, identify its authors, and then determine who the youngest author is. The paper is likely "Attention Is All You Need" (2017).

Action 1
<search>Attention Is All You Need</search>|xml_attributes_break|


Après la "Thought" et l'"Action" générées par le modèle, c'est à vous de fournir l'"Observation" (le résultat de l'action). Cette cellule simule l'observation d'une recherche sur un papier NLP, et le modèle est censé utiliser cette information pour continuer son raisonnement.

Vous pouvez maintenant faire cette recherche vous-même et fournir le résultat au modèle.

In [27]:
# Simulation du résultat d'une recherche externe (Observation 1) que nous injectons à l'agent
observation = """Observation 1
[1706.03762] Attention Is All You Need
Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
"""

# Envoi de l'observation pour obtenir l'étape de pensée suivante
resp = react_chat.send_message(observation)
print(resp.text)

Thought 2
The authors are Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin. I will search for Aidan N. Gomez to check his birth year, as he was an undergraduate/intern at the time and is widely cited as the youngest author on the paper.

Action 2
<search>Aidan Gomez</search>|xml_attributes_break|


Cette section introduit le "mode réflexion" des modèles Gemini. Les modèles comme Gemini Flash 2.0 (et les modèles Gemini 3 qui l'intègrent par défaut) peuvent générer en interne un processus de pensée ("thinking process") pour arriver à une meilleure réponse, sans nécessiter de prompting CoT explicite. Bien que les pensées intermédiaires ne soient pas directement exposées par l'API dans la réponse finale, ce mode améliore le raisonnement.

Ce processus se répète jusqu'à ce que l'action `<finish>` soit atteinte. Vous pouvez continuer vous-même si vous le souhaitez, ou essayer l'[exemple Wikipédia](https://github.com/google-gemini/cookbook/blob/main/examples/Search_Wikipedia_using_ReAct.ipynb) pour voir un système ReAct entièrement automatisé à l'œuvre.

## Mode réflexion (thinking mode)

Le modèle expérimental Gemini Flash 2.0 « Thinking » a été entraîné à générer le « processus de réflexion » qu'il suit dans le cadre de sa réponse. Le modèle Flash Thinking offre ainsi de meilleures capacités de raisonnement.

Utiliser un modèle en « mode réflexion » permet d'obtenir des réponses de grande qualité sans prompting spécialisé comme dans les approches précédentes. Si cette technique est efficace, c'est notamment parce qu'elle amène le modèle à générer des informations pertinentes (« brainstorming » ou « pensées ») qui servent ensuite de contexte pour produire la réponse finale.

Notez qu'avec l'API, vous obtenez la réponse finale du modèle, mais les pensées ne sont pas renvoyées. Pour voir les pensées intermédiaires, essayez [le modèle en mode réflexion dans AI Studio](https://aistudio.google.com/prompts/new_chat?model=gemini-2.0-flash-thinking-exp-01-21).

> **Mise à jour** : le modèle `gemini-2.0-flash-thinking-exp` n'existe plus. Les modèles Gemini 3 intègrent la réflexion par défaut : la cellule ci-dessous utilise donc simplement `MODEL`.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1Z991SV7lZZZqioOiqIUPv9a9ix-ws4zk"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

Cette cellule utilise le mode streaming (`generate_content_stream`) pour afficher la réponse au fur et à mesure qu'elle est générée, ce qui peut être utile pour les réponses longues. Le modèle est interrogé sur l'auteur le plus jeune d'un papier NLP, tirant parti de ses capacités de raisonnement intégrées.

In [28]:
import io
from IPython.display import Markdown, clear_output

# Exemple d'appel en Streaming : la réponse s'affiche mot par mot à l'écran
response = client.models.generate_content_stream(
    model=MODEL,
    contents='Who was the youngest author listed on the transformers NLP paper?',
)

buf = io.StringIO()
for chunk in response:
    if chunk.text:
        buf.write(chunk.text)
        # Affiche le flux brut au fur et à mesure
        print(chunk.text, end='')

# Remplacement final de la console brute par un rendu visuel Markdown propre
clear_output()
display(Markdown(buf.getvalue()))

The youngest author of the seminal Transformers paper, *"Attention Is All You Need"* (published in 2017), is **Aidan Gomez** (Aidan N. Gomez). 

At the time the research was conducted and published, Gomez was an undergraduate student at the University of Toronto doing an internship at Google Brain. He was only about **21 years old** when the paper was published. 

He went on to complete his PhD at the University of Oxford under Yarin Gal and Geoffrey Hinton, and later co-founded the prominent AI company **Cohere**, where he currently serves as CEO.

Cette section se concentre sur les capacités de génération et d'explication de code des modèles Gemini.

## Prompting pour le code

Les modèles Gemini peuvent générer du code dans divers langages. Cette cellule illustre la génération d'une fonction Python pour calculer la factorielle d'un nombre, avec une instruction explicite de ne fournir que le code, sans explication supplémentaire.

### Générer du code

Les modèles de la famille Gemini peuvent générer du code, des fichiers de configuration et des scripts. C'est utile pour apprendre à programmer, découvrir un nouveau langage ou produire rapidement un premier jet.

Comme les LLM peuvent se tromper et reproduire des données d'entraînement, il est indispensable de lire et de tester le code avant de l'utiliser, et de respecter les licences applicables.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1YX71JGtzDjXQkgdes8bP6i3oH5lCRKxv"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [29]:
from google.genai import types

# Définition d'un prompt pour la génération de code sans explications superflues
code_prompt = """
Write a Python function to calculate the factorial of a number. No explanation, provide only the code.
"""

response = client.models.generate_content(
    model=MODEL,
    config=types.GenerateContentConfig(
        temperature=1,
        top_p=1,
        max_output_tokens=1024,
    ),
    contents=code_prompt
)

Markdown(response.text)

```python
def factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result
```

Une fonctionnalité puissante de l'API Gemini est sa capacité à exécuter le code qu'elle génère. Cette section montre comment utiliser les `tools` (`code_execution`) pour demander au modèle de générer une liste de nombres premiers impairs et d'en calculer la somme, puis d'exécuter ce code et de renvoyer le résultat.

### Exécution de code

L'API Gemini peut aussi exécuter automatiquement le code généré et renvoyer le résultat.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/11veFr_VYEwBWcLkhNLr-maCG0G8sS_7Z"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [30]:
from pprint import pprint
from google.genai import types

# Configuration de l'exécution de code intégrée (Sandboxed Code Execution)
config = types.GenerateContentConfig(
    tools=[types.Tool(code_execution=types.ToolCodeExecution())],
)

code_exec_prompt = """
Generate the first 14 odd prime numbers, then calculate their sum.
"""

response = client.models.generate_content(
    model=MODEL,
    config=config,
    contents=code_exec_prompt
)

# Inspection structurée de toutes les parties générées (Pensée, code, résultat de l'exécution de code)
for part in response.candidates[0].content.parts:
  pprint(part.to_json_dict())
  print("-----")

{'executable_code': {'code': 'def is_prime(n):\n'
                             '    if n < 2:\n'
                             '        return False\n'
                             '    for i in range(2, int(n**0.5) + 1):\n'
                             '        if n % i == 0:\n'
                             '            return False\n'
                             '    return True\n'
                             '\n'
                             'odd_primes = []\n'
                             'num = 3 # Start at 3 (first odd prime)\n'
                             'while len(odd_primes) < 14:\n'
                             '    if is_prime(num):\n'
                             '        odd_primes.append(num)\n'
                             '    num += 2 # Skip even numbers\n'
                             '\n'
                             'sum_primes = sum(odd_primes)\n'
                             'print(f"First 14 odd primes: {odd_primes}")\n'
                             'print(f"S

Lorsque le mode d'exécution de code est activé, la réponse du modèle peut contenir plusieurs parties : texte, code exécutable et résultats de l'exécution. Cette cellule parcourt les différentes parties de la réponse et les affiche de manière appropriée pour présenter le texte, le code généré et l'output de son exécution.

Cette réponse contient plusieurs parties : une partie texte d'ouverture et une de clôture, qui correspondent à des réponses classiques, une partie `executable_code` qui contient le code généré, et une partie `code_execution_result` qui contient le résultat de son exécution.

Vous pouvez les explorer une par une.

In [31]:
# Analyse et affichage dynamique des différentes parties (textes descriptifs, blocs de code, et sorties de script d'exécution)
for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    elif part.executable_code:
        display(Markdown(f'```python\n{part.executable_code.code}\n```'))
    elif part.code_execution_result:
        # Si le script a crashé ou échoué, on affiche son état technique d'erreur
        if part.code_execution_result.outcome != 'OUTCOME_OK':
            display(Markdown(f'## Status {part.code_execution_result.outcome}'))
        # Affichage du résultat de sortie de console généré par l'exécution du code
        display(Markdown(f'```\n{part.code_execution_result.output}\n```'))

```python
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

odd_primes = []
num = 3 # Start at 3 (first odd prime)
while len(odd_primes) < 14:
    if is_prime(num):
        odd_primes.append(num)
    num += 2 # Skip even numbers

sum_primes = sum(odd_primes)
print(f"First 14 odd primes: {odd_primes}")
print(f"Sum: {sum_primes}")

```

```
First 14 odd primes: [3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Sum: 326

```

The first 14 odd prime numbers are:

1. **3**
2. **5**
3. **7**
4. **11**
5. **13**
6. **17**
7. **19**
8. **23**
9. **29**
10. **31**
11. **37**
12. **41**
13. **43**
14. **47**

*(Note: The number 2 is excluded because it is an even prime number).*

### Calculation of their sum:
$$\text{Sum} = 3 + 5 + 7 + 11 + 13 + 17 + 19 + 23 + 29 + 31 + 37 + 41 + 43 + 47 = 326$$

The sum of the first 14 odd prime numbers is **326**.

Les modèles Gemini peuvent également expliquer du code existant. Cette cellule télécharge un script Bash (`gitprompt.sh`) et demande au modèle d'en fournir une explication de haut niveau, décrivant son utilité et son fonctionnement général.

### Expliquer du code

Les modèles de la famille Gemini peuvent aussi vous expliquer du code. Dans cet exemple, vous leur transmettez un [script bash](https://github.com/magicmonty/bash-git-prompt) et posez quelques questions.

<table align=left>
  <td>
    <a target="_blank" href="https://aistudio.google.com/prompts/1N7LGzWzCYieyOf_7bAG4plrmkpDNmUyb"><img src="https://ai.google.dev/site-assets/images/marketing/home/icon-ais.png" style="height: 24px" height=24/> Ouvrir dans AI Studio</a>
  </td>
</table>

In [32]:
# Récupération en ligne du contenu d'un script bash externe réel via curl
file_contents = !curl -s https://raw.githubusercontent.com/magicmonty/bash-git-prompt/refs/heads/master/gitprompt.sh

# Demande d'explication pédagogique globale du fonctionnement du script récupéré
explain_prompt = f"""
Please explain what this file does at a very high level. What is it, and why would I use it?

```
{file_contents}
```
"""

response = client.models.generate_content(
    model=MODEL,
    contents=explain_prompt
)

Markdown(response.text)

At a very high level, this file is the core engine of **`bash-git-prompt`**, a highly popular open-source script designed to customize your command-line prompt (the text you see in your terminal before you type a command). 

It is compatible with both **Bash** and **Zsh** shells.

---

### What is it?
It is a terminal helper. Whenever you navigate into a folder on your computer that is a Git repository, this script automatically detects it and rebuilds your terminal prompt to show you the exact status of your repository. 

### Why would I use it?
You would use this to **save time and stay oriented** when working with Git. Instead of constantly typing `git status` or `git branch` to see what is going on, this script displays that information right in your prompt, in real-time.

Here is what it does for you automatically:
* **Shows your current branch:** Tells you exactly which Git branch you are currently on.
* **Displays Git status indicators:** Shows you at a glance how many files are:
  * 🟢 Staged (ready to commit)
  * 🟡 Changed/modified (not yet staged)
  * 🔴 Conflicted (merge conflicts)
  * ⚪ Untracked (new files)
  * 📦 Stashed (saved for later)
* **Tracks remote sync status:** Shows if your local branch is **ahead** of or **behind** the remote repository (e.g., GitHub), and even runs `git fetch` silently in the background so this information stays up to date.
* **Integrates with Virtual Environments:** If you are a programmer using Python (Conda/Virtualenv) or Node.js, it will display your active virtual environment in the prompt as well.
* **Highly Customizable:** It supports themes, allowing you to change the colors, symbols, and layout of the prompt.

### How it works (under the hood)
It hooks into your shell's layout generator (`PROMPT_COMMAND` in Bash). Every time you run a command or press Enter, this script runs quickly in the background, checks if you are in a Git project, queries Git for the status, and styles your prompt accordingly.

Cette dernière section fournit des ressources supplémentaires pour approfondir vos connaissances en matière de "prompting" et d'utilisation de l'API Gemini, y compris la documentation officielle, des galeries de prompts et des exemples pratiques.

## Pour aller plus loin

Pour approfondir le prompting :

* Lisez l'[introduction au prompting](https://ai.google.dev/gemini-api/docs/prompting-intro) dans la documentation de l'API Gemini ;
* Explorez la [galerie de prompts](https://ai.google.dev/gemini-api/prompts) de l'API Gemini et testez-les dans AI Studio ;
* Parcourez le cookbook de l'API Gemini pour des [exemples inspirants](https://github.com/google-gemini/cookbook/blob/main/examples/) et des [guides de démarrage pédagogiques](https://github.com/google-gemini/cookbook/blob/main/quickstarts/).
